In [188]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 32    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3 * 0.6
WD = 1e-4
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 5
PATIENCE = 12 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

# --- added: CV alpha sweep + refit ---
MODEL_SEEDS = [6, 7, 8]
ALPHA_GRID = [0.7, 0.75, 0.778, 0.82, 0.86]
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns

In [189]:
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }

In [190]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))

Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [ ]:
val_targets = df_valmap["pert"].astype(str).tolist()

union_genes = sorted(set([g.upper() for g in gene_columns] +
                         [g.upper() for g in train_genes.tolist()] +
                         [g.upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

# (80, 5127) dense -> signed log transform to handle negatives
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

# SVD across genes: components_ is (k, 5127) so transpose to (5127, k)
svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)

# Map output genes to embeddings
gene2emb_pert = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_PERT].copy()
                 for i in range(len(gene_columns))}
gene2emb_out  = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_OUT ].copy()
                 for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

missing_emb_pert = {}

def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()
    if gU in gene2emb_pert:
        return gene2emb_pert[gU]
    if gU in missing_emb_pert:
        return missing_emb_pert[gU]
    return emb_fallback_pert

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

# Output gene embeddings in the exact output gene order
U_out = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)    # (G, d_out)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (80, d_pert)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

# Optional: visibility into coverage
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val = [g for g in val_targets if str(g).upper() not in geneU]
if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")


# h5ad for missing perts: embed by control-cell coexpression
missing_all = sorted(set([str(x).upper() for x in (missing_train + missing_val)]))

if len(missing_all) > 0:
    try:
        import scipy.sparse as sp
        import anndata as ad

        print("[h5ad] building embeddings for missing perts:", len(missing_all))
        adata = ad.read_h5ad(str(H5AD_PATH))

        # pick perturbation column
        pert_col = None
        for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
            if c in adata.obs.columns:
                pert_col = c
                break
        if pert_col is None:
            raise ValueError("Could not find perturbation column in h5ad obs.")

        Xc = adata.X
        if not sp.issparse(Xc):
            Xc = sp.csr_matrix(Xc)
        else:
            Xc = Xc.tocsr()

        # normalize ALL genes: CPM10K then log2(1+x)
        cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
        scale = (10000.0 / cell_sum).astype(np.float64)

        Xn = Xc.multiply(scale[:, None]).tocsr()
        Xn.data = np.log1p(Xn.data) / np.log(2.0)

        ctrl_mask = (adata.obs[pert_col].astype(str).to_numpy() == "non-targeting")
        if int(ctrl_mask.sum()) == 0:
            raise ValueError("No non-targeting control cells found in h5ad.")

        var = {str(g).upper(): i for i, g in enumerate(adata.var_names.astype(str).to_numpy())}

        # indices for the 5127 output genes in the 19,226 gene space
        out_idx = np.array([var[str(g).upper()] for g in gene_columns if str(g).upper() in var], dtype=np.int64)
        if len(out_idx) != len(gene_columns):
            miss_out = [g for g in gene_columns if str(g).upper() not in var]
            raise ValueError(f"{len(miss_out)} output genes missing from h5ad var_names (unexpected). Example: {miss_out[:10]}")

        Xout = Xn[ctrl_mask][:, out_idx]  # (n_ctrl, 5127)
        if sp.issparse(Xout):
            Xout = Xout.toarray()
        Xout = Xout.astype(np.float32)

        mu = Xout.mean(axis=0, keepdims=True)
        sd = Xout.std(axis=0, keepdims=True) + 1e-6
        Xout_z = (Xout - mu) / sd

        # use the *existing* pert embedding space (from SVD on D_train): (5127, d_pert)
        P_out = gene_emb_all[:, :EMB_DIM_PERT].astype(np.float32)

        topk = 256
        made = 0
        for gU in missing_all:
            if gU not in var:
                continue

            xg = Xn[ctrl_mask, var[gU]]
            if sp.issparse(xg):
                xg = xg.toarray()
            xg = np.asarray(xg).ravel().astype(np.float32)

            xg = (xg - xg.mean()) / (xg.std() + 1e-6)

            corr = (xg[:, None] * Xout_z).mean(axis=0)  # (5127,)
            idx = np.argsort(-np.abs(corr))[:topk]
            w = corr[idx].astype(np.float32)

            z = (w[:, None] * P_out[idx]).sum(axis=0)
            z = z / (np.linalg.norm(z) + 1e-12)

            missing_emb_pert[gU] = z.astype(np.float32)
            made += 1

        print("[h5ad] embedded missing perts:", made, "of", len(missing_all))
    except Exception as e:
        print("[h5ad] skipped missing pert embeddings due to error:", repr(e))

SVD k: 80 gene_emb_all: (5127, 80)
U_out: (5127, 80) Z_train: (80, 80)
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] building embeddings for missing perts: 16
[h5ad] embedded missing perts: 16 of 16


In [ ]:
def gate_smoothstep(x, a = GATE_A, b = GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    """
    Same logic as weighted_l1_like, but returns (N,) per-row.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

import torch

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Row-weighted version of existing weighted_l1_like.

    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

def weighted_cosine_per_row_torch(dt: torch.Tensor, dp: torch.Tensor, w: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # dt, dp, w: (B, G)
    wa = w * dt
    wb = w * dp
    num = torch.sum(wa * wb, dim=1)
    da  = torch.sqrt(torch.sum(wa * wa, dim=1))
    db  = torch.sqrt(torch.sum(wb * wb, dim=1))
    denom = torch.clamp(da * db, min=eps)
    return num / denom  # (B,)


In [193]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device) # (G, d_out)
Zt = torch.tensor(Z_train, device=device) # (N, d_pert)
Yt = torch.tensor(Y, device=device) # (N, G)

delta_baseline_vec = D_train.mean(axis=0).astype(np.float32)
delta_baseline = np.tile(delta_baseline_vec[None, :], (N, 1))

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [194]:
gt_df = pd.read_csv(ROOT / "data" / "training_data_ground_truth_table.csv")

bw_map = gt_df.groupby("pert_id")["baseline_wmae"].mean()

baseline_wmae = bw_map.reindex(df_train["pert_symbol"].astype(str)).to_numpy(np.float32)

assert baseline_wmae.shape[0] == N, (baseline_wmae.shape, N)
assert np.all(np.isfinite(baseline_wmae)), "baseline_wmae has NaNs after alignment"

baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

## Add STRING v12 embeddings (sequence + network)

This section loads STRING embeddings and evaluates a gated hybrid variant of the same bilinear model.

It keeps:
- same folds
- same loss
- same alpha sweep
- same scoring

It only changes the model input by adding STRING feature blocks with a learned gate.

In [ ]:
import json
import h5py
import re

STRING_DIR = ROOT / "external" / "string"
ALIASES_PATH = STRING_DIR / "9606.protein.aliases.v12.0.txt"
SEQ_H5_PATH  = STRING_DIR / "9606.protein.sequence.embeddings.v12.0.h5"
NET_H5_PATH  = STRING_DIR / "9606.protein.network.embeddings.v12.0.h5"

for p in [ALIASES_PATH, SEQ_H5_PATH, NET_H5_PATH]:
    print(("OK " if p.exists() else "MISSING "), p)

CACHE_DIR = STRING_DIR / "_cache_myllia"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

needed_genesU = sorted(set([str(g).upper() for g in gene_columns] +
                           [str(g).upper() for g in train_genes.tolist()] +
                           [str(g).upper() for g in df_valmap["pert"].astype(str).tolist()]))

print("needed gene symbols:", len(needed_genesU))

MAP_CACHE = CACHE_DIR / f"gene2prot_needed_{len(needed_genesU)}.json"

def looks_like_gene_symbol(s: str) -> bool:
    if s is None:
        return False
    if len(s) < 2 or len(s) > 20:
        return False
    if not re.fullmatch(r"[A-Z0-9\.\-]+", s):
        return False
    if re.fullmatch(r"\d+", s):
        return False
    return True

def build_gene2prot_streaming(aliases_path: Path, needed_set: set) -> dict:
    # Filter by alias shape and require exact match to our needed set.
    gene2prot = {g: set() for g in needed_set}

    hits = 0
    seen = 0
    with open(aliases_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line or line[0] == "#":
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            prot, alias, src = parts[0], parts[1], parts[2]
            aliasU = alias.upper()
            if aliasU in gene2prot and looks_like_gene_symbol(aliasU):
                gene2prot[aliasU].add(prot)
                hits += 1
            seen += 1
            if seen % 5_000_000 == 0:
                print(f"[aliases] lines={seen:,} hits={hits:,}")

    gene2prot = {g: sorted(list(v)) for g, v in gene2prot.items() if len(v) > 0}
    return gene2prot

if MAP_CACHE.exists():
    gene2prot = json.loads(MAP_CACHE.read_text(encoding="utf-8"))
    print("[cache] loaded gene2prot:", len(gene2prot), "from", MAP_CACHE)
else:
    gene2prot = build_gene2prot_streaming(ALIASES_PATH, set(needed_genesU))
    MAP_CACHE.write_text(json.dumps(gene2prot), encoding="utf-8")
    print("[cache] wrote gene2prot:", len(gene2prot), "to", MAP_CACHE)

mapped_needed = set(gene2prot.keys())
missing_needed = [g for g in needed_genesU if g not in mapped_needed]
print("mapped needed:", len(mapped_needed), "missing needed:", len(missing_needed))
print("missing examples:", missing_needed[:20])

# =============================
# Rescue pass v2 for missing genes
# - Fixes versioned symbols: C4ORF36.1 -> C4ORF36
# - Optionally tries safe hyphen splits for readthrough-ish names
# Notes:
# - VEGFA is genuinely absent from STRING v12 human alias set (so it will stay missing).
# =============================
import re as _re

def _canon_gene(s: str) -> str:
    s = str(s).upper().strip()
    if ":" in s:
        s = s.split(":")[-1]
    # strip trailing .<digits> (e.g., C4ORF36.1)
    s = _re.sub(r"\.\d+$", "", s)
    return s

def _norm_key(s: str) -> str:
    s = _canon_gene(s)
    # remove separators that often differ between sources
    s = s.replace(" ", "").replace("_", "").replace("-", "")
    return s

def _candidate_keys_for_missing(g: str, try_hyphen_split: bool = False):
    g0 = _canon_gene(g)
    cands = [g0]
    if try_hyphen_split and "-" in g0:
        parts = [p for p in g0.split("-") if p]
        # only include parts that look like "real" gene-ish tokens
        cands += [p for p in parts if looks_like_gene_symbol(p)]
    keys = [_norm_key(x) for x in cands]
    # unique, preserve order
    out = []
    seen = set()
    for k in keys:
        if k not in seen:
            out.append(k)
            seen.add(k)
    return out

missing_before = list(missing_needed)
print("[rescue] missing before:", len(missing_before), "example:", missing_before[:10])

RESCUE_CACHE = CACHE_DIR / f"gene2prot_needed_{len(needed_genesU)}_rescue_v2.json"

if RESCUE_CACHE.exists():
    gene2prot = json.loads(RESCUE_CACHE.read_text(encoding="utf-8"))
    mapped_needed = set(gene2prot.keys())
    missing_needed = [g for g in needed_genesU if g not in mapped_needed]
    print("[rescue] loaded:", RESCUE_CACHE, "| mapped:", len(mapped_needed), "missing:", len(missing_needed))
else:
    if len(missing_before) > 0:
        # Map normalized keys -> original missing gene symbol; drop ambiguous keys
        norm_to_gene = {}
        collisions = set()

        TRY_HYPHEN_SPLIT = False

        for g in missing_before:
            for k in _candidate_keys_for_missing(g, try_hyphen_split=TRY_HYPHEN_SPLIT):
                if k in norm_to_gene and norm_to_gene[k] != g:
                    collisions.add(k)
                else:
                    norm_to_gene[k] = g

        for k in collisions:
            norm_to_gene.pop(k, None)

        gene2prot2 = {g: set(pids) for g, pids in gene2prot.items()}
        for g in missing_before:
            gene2prot2.setdefault(g, set())

        hits = 0
        seen = 0
        with open(ALIASES_PATH, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if not line or line[0] == "#":
                    continue
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 3:
                    continue
                prot, alias, src = parts[0], parts[1], parts[2]
                k = _norm_key(alias)
                g = norm_to_gene.get(k)
                if g is not None:
                    gene2prot2[g].add(prot)
                    hits += 1
                seen += 1
                if seen % 5_000_000 == 0:
                    print(f"[rescue aliases] lines={seen:,} hits={hits:,}")

        gene2prot = {g: sorted(list(v)) for g, v in gene2prot2.items() if len(v) > 0}
        RESCUE_CACHE.write_text(json.dumps(gene2prot), encoding="utf-8")
        print("[rescue] wrote:", RESCUE_CACHE)

    mapped_needed = set(gene2prot.keys())
    missing_needed = [g for g in needed_genesU if g not in mapped_needed]
    print("[rescue] mapped after:", len(mapped_needed), "missing after:", len(missing_needed))

print("[rescue] VEGFA mapped?", "VEGFA" in set(gene2prot.keys()), "->", gene2prot.get("VEGFA", [])[:5])

# write missing list AFTER rescue
pd.DataFrame({"missing_gene_symbol": missing_needed}).to_csv(CACHE_DIR/"missing_needed_gene_symbols.csv", index=False)


OK  external\string\9606.protein.aliases.v12.0.txt
OK  external\string\9606.protein.sequence.embeddings.v12.0.h5
OK  external\string\9606.protein.network.embeddings.v12.0.h5
needed gene symbols: 5143
[cache] loaded gene2prot: 5054 from external\string\_cache_myllia\gene2prot_needed_5143.json
mapped needed: 5054 missing needed: 89
missing examples: ['ABTB3', 'ADISSP', 'AFG2A', 'AQP7B', 'ATOSA', 'ATOSB', 'ATP5MJ', 'BBLN', 'BORCS7-ASMT', 'C1QTNF3-AMACR', 'C4ORF36.1', 'CCDC92B', 'CEP15', 'CFAP68', 'CFAP95', 'CFAP96', 'CIMAP1C', 'CMKLR2', 'CYP2D7', 'DNAAF8']
[rescue] missing before: 89 example: ['ABTB3', 'ADISSP', 'AFG2A', 'AQP7B', 'ATOSA', 'ATOSB', 'ATP5MJ', 'BBLN', 'BORCS7-ASMT', 'C1QTNF3-AMACR']
[rescue] wrote: external\string\_cache_myllia\gene2prot_needed_5143_rescue_v2.json
[rescue] mapped after: 5056 missing after: 87
[rescue] VEGFA mapped? False -> []


In [206]:
def decode_obj_bytes(arr):
    if arr is None:
        return None
    out = []
    for x in arr:
        if isinstance(x, (bytes, bytearray)):
            out.append(x.decode("utf-8"))
        else:
            out.append(str(x))
    return np.array(out, dtype=object)

def load_string_h5(path: Path):
    with h5py.File(path, "r") as f:
        ds_names = []
        def walk(name, obj):
            if isinstance(obj, h5py.Dataset):
                ds_names.append(name)
        f.visititems(walk)

        # protein id dataset
        prot_key = None
        for name in ds_names:
            ds = f[name]
            if ds.ndim == 1 and ds.dtype.kind in ("S","O","U"):
                try:
                    v0 = ds[0]
                except Exception:
                    continue
                if isinstance(v0, (bytes, bytearray)) and (b"9606." in v0) and (b"ENSP" in v0):
                    prot_key = name
                    break
        if prot_key is None:
            for name in ds_names:
                ds = f[name]
                if ds.ndim == 1 and ds.dtype.kind in ("S","O","U"):
                    prot_key = name
                    break
        if prot_key is None:
            raise ValueError(f"Could not find protein id dataset in {path}")

        prot = decode_obj_bytes(f[prot_key][:])

        # embedding dataset matching first dim
        emb_key = None
        for name in ds_names:
            ds = f[name]
            if ds.ndim == 2 and ds.dtype.kind in ("f","i") and ds.shape[0] == len(prot):
                emb_key = name
                break
        if emb_key is None:
            best = None
            best_n = -1
            for name in ds_names:
                ds = f[name]
                if ds.ndim == 2 and ds.dtype.kind in ("f","i"):
                    n = ds.shape[0] * ds.shape[1]
                    if n > best_n:
                        best_n = n
                        best = name
            emb_key = best
        if emb_key is None:
            raise ValueError(f"Could not find embedding dataset in {path}")

        emb = np.asarray(f[emb_key][:], dtype=np.float32)

    return prot, emb, {"prot_key": prot_key, "emb_key": emb_key, "shape": emb.shape}

seq_prot, seq_emb_all, seq_info = load_string_h5(SEQ_H5_PATH)
net_prot, net_emb_all, net_info = load_string_h5(NET_H5_PATH)

print("SEQ:", seq_info, "prot sample:", seq_prot[:3])
print("NET:", net_info, "prot sample:", net_prot[:3])

seq_dim = int(seq_emb_all.shape[1])
net_dim = int(net_emb_all.shape[1])
print("seq_dim:", seq_dim, "net_dim:", net_dim)

seq_idx = {pid: i for i, pid in enumerate(seq_prot.tolist())}
net_idx = {pid: i for i, pid in enumerate(net_prot.tolist())}

SEQ: {'prot_key': 'proteins', 'emb_key': 'embeddings', 'shape': (19699, 1024)} prot sample: ['9606.ENSP00000000233' '9606.ENSP00000000412' '9606.ENSP00000001008']
NET: {'prot_key': 'proteins', 'emb_key': 'embeddings', 'shape': (19699, 512)} prot sample: ['9606.ENSP00000497020' '9606.ENSP00000498128' '9606.ENSP00000497439']
seq_dim: 1024 net_dim: 512


In [198]:
def mean_emb_for_gene(gU: str, gene2prot: dict, idx_map: dict, emb_all: np.ndarray):
    pids = gene2prot.get(gU)
    if not pids:
        return None
    rows = [idx_map.get(pid) for pid in pids]
    rows = [r for r in rows if r is not None]
    if len(rows) == 0:
        return None
    return emb_all[rows].mean(axis=0).astype(np.float32)

def l2_normalize_vec(v, eps=1e-8):
    n = float(np.linalg.norm(v))
    if n < eps:
        return v
    return (v / (n + eps)).astype(np.float32)

U_seq = np.zeros((len(gene_columns), seq_dim), dtype=np.float32)
U_net = np.zeros((len(gene_columns), net_dim), dtype=np.float32)
U_seq_mask = np.zeros((len(gene_columns), 1), dtype=np.float32)
U_net_mask = np.zeros((len(gene_columns), 1), dtype=np.float32)

seq_hit = 0
net_hit = 0
for i, g in enumerate(gene_columns):
    gU = str(g).upper()
    vs = mean_emb_for_gene(gU, gene2prot, seq_idx, seq_emb_all)
    vn = mean_emb_for_gene(gU, gene2prot, net_idx, net_emb_all)

    if vs is not None:
        vs = l2_normalize_vec(vs)
        U_seq[i] = vs
        U_seq_mask[i, 0] = 1.0
        seq_hit += 1

    if vn is not None:
        vn = l2_normalize_vec(vn)
        U_net[i] = vn
        U_net_mask[i, 0] = 1.0
        net_hit += 1

print("[U] genes:", len(gene_columns), "seq_hit:", seq_hit, "net_hit:", net_hit)

Z_seq = np.zeros((len(train_genes), seq_dim), dtype=np.float32)
Z_net = np.zeros((len(train_genes), net_dim), dtype=np.float32)
Z_seq_mask = np.zeros((len(train_genes), 1), dtype=np.float32)
Z_net_mask = np.zeros((len(train_genes), 1), dtype=np.float32)

seq_hit_p = 0
net_hit_p = 0
for i, g in enumerate(train_genes.tolist()):
    gU = str(g).upper()
    vs = mean_emb_for_gene(gU, gene2prot, seq_idx, seq_emb_all)
    vn = mean_emb_for_gene(gU, gene2prot, net_idx, net_emb_all)

    if vs is not None:
        vs = l2_normalize_vec(vs)
        Z_seq[i] = vs
        Z_seq_mask[i, 0] = 1.0
        seq_hit_p += 1

    if vn is not None:
        vn = l2_normalize_vec(vn)
        Z_net[i] = vn
        Z_net_mask[i, 0] = 1.0
        net_hit_p += 1

print("[Z] perts:", len(train_genes), "seq_hit:", seq_hit_p, "net_hit:", net_hit_p)
print("[coverage] U_seq:", float(U_seq_mask.mean()), "U_net:", float(U_net_mask.mean()),
      "Z_seq:", float(Z_seq_mask.mean()), "Z_net:", float(Z_net_mask.mean()))

[U] genes: 5127 seq_hit: 5039 net_hit: 5039
[Z] perts: 80 seq_hit: 79 net_hit: 79
[coverage] U_seq: 0.9828359484672546 U_net: 0.9828359484672546 Z_seq: 0.987500011920929 Z_net: 0.987500011920929


In [207]:
U_seq_t = torch.tensor(U_seq, device=device, dtype=torch.float32)
U_net_t = torch.tensor(U_net, device=device, dtype=torch.float32)
Z_seq_t = torch.tensor(Z_seq, device=device, dtype=torch.float32)
Z_net_t = torch.tensor(Z_net, device=device, dtype=torch.float32)

U_seq_mask_t = torch.tensor(U_seq_mask, device=device, dtype=torch.float32)
U_net_mask_t = torch.tensor(U_net_mask, device=device, dtype=torch.float32)
Z_seq_mask_t = torch.tensor(Z_seq_mask, device=device, dtype=torch.float32)
Z_net_mask_t = torch.tensor(Z_net_mask, device=device, dtype=torch.float32)

In [208]:
class HybridStringBilinearDeltaModel(nn.Module):
    # Bilinear predictor with 3 blocks each side:
    # base, STRING seq, STRING net
    # Fixes:
    # - bias=False on seq/net Linear so missing zeros do not turn into learned constants
    # - gate logits masked so missing blocks cannot be selected by softmax

    def __init__(self, d_base_p, d_base_o, d_seq, d_net, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        self.p_base = nn.Sequential(nn.Linear(d_base_p, rank_r), nn.GELU(), nn.Dropout(dropout))
        self.p_seq  = nn.Sequential(nn.Linear(d_seq, rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))
        self.p_net  = nn.Sequential(nn.Linear(d_net, rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))

        self.o_base = nn.Sequential(nn.Linear(d_base_o, rank_r), nn.GELU(), nn.Dropout(dropout))
        self.o_seq  = nn.Sequential(nn.Linear(d_seq, rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))
        self.o_net  = nn.Sequential(nn.Linear(d_net, rank_r, bias=False), nn.GELU(), nn.Dropout(dropout))

        self.gate_p = nn.Sequential(
            nn.Linear(3 * rank_r, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3),
        )
        self.gate_o = nn.Sequential(
            nn.Linear(3 * rank_r, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3),
        )

        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = nn.Parameter(torch.zeros(1, device=dev))

    @staticmethod
    def _mask_logits_3(logits, m_seq, m_net, big=1e9):
        # logits: (B,3), masks: (B,1) float 0/1
        # If mask is 0, push that logit to a huge negative value so softmax ignores it.
        # Base block is always available.
        if m_seq is not None:
            logits = logits.clone()
            logits[:, 1] = logits[:, 1] + (m_seq.squeeze(1) - 1.0) * big
        if m_net is not None:
            logits = logits.clone()
            logits[:, 2] = logits[:, 2] + (m_net.squeeze(1) - 1.0) * big
        return logits

    def forward(
        self,
        z_base, z_seq, z_net,
        u_base, u_seq, u_net,
        z_seq_mask=None, z_net_mask=None,
        u_seq_mask=None, u_net_mask=None
    ):
        pb = self.p_base(z_base)
        ps = self.p_seq(z_seq)
        pn = self.p_net(z_net)

        ob = self.o_base(u_base)
        os = self.o_seq(u_seq)
        on = self.o_net(u_net)

        # Default masks to "present" if not provided
        if z_seq_mask is None:
            z_seq_mask = torch.ones((z_base.shape[0], 1), device=z_base.device, dtype=z_base.dtype)
        if z_net_mask is None:
            z_net_mask = torch.ones((z_base.shape[0], 1), device=z_base.device, dtype=z_base.dtype)
        if u_seq_mask is None:
            u_seq_mask = torch.ones((u_base.shape[0], 1), device=u_base.device, dtype=u_base.dtype)
        if u_net_mask is None:
            u_net_mask = torch.ones((u_base.shape[0], 1), device=u_base.device, dtype=u_base.dtype)

        gp_logits = self.gate_p(torch.cat([pb, ps, pn], dim=1))
        gp_logits = self._mask_logits_3(gp_logits, z_seq_mask, z_net_mask)
        gp = torch.softmax(gp_logits, dim=1)

        go_logits = self.gate_o(torch.cat([ob, os, on], dim=1))
        go_logits = self._mask_logits_3(go_logits, u_seq_mask, u_net_mask)
        go = torch.softmax(go_logits, dim=1)

        p = gp[:, 0:1] * pb + gp[:, 1:2] * ps + gp[:, 2:3] * pn
        o = go[:, 0:1] * ob + go[:, 1:2] * os + go[:, 2:3] * on

        y = p @ o.T
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [209]:
def apply_shrink(pred, baseline_bg, alpha):
    a = float(alpha)
    return a * pred + (1.0 - a) * baseline_bg

In [210]:
def train_one_fold_string(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridStringBilinearDeltaModel(
        d_base_p=Zt.shape[1],
        d_base_o=Uo_t.shape[1],
        d_seq=U_seq_t.shape[1],
        d_net=U_net_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)

    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)
    va_true = Y[va_idx]                   # (B,G) numpy
    va_base = delta_baseline[va_idx]      # (B,G) numpy

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_raw = None   # (B,G) numpy RAW (no shrink)
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(
                Zt.index_select(0, b_t),
                Z_seq_t.index_select(0, b_t),
                Z_net_t.index_select(0, b_t),
                Uo_t,
                U_seq_t,
                U_net_t,
                Z_seq_mask_t.index_select(0, b_t),
                Z_net_mask_t.index_select(0, b_t),
                U_seq_mask_t,
                U_net_mask_t,
            )

            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 5 == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(
                    Zt.index_select(0, va_idx_t),
                    Z_seq_t.index_select(0, va_idx_t),
                    Z_net_t.index_select(0, va_idx_t),
                    Uo_t,
                    U_seq_t,
                    U_net_t,
                    Z_seq_mask_t.index_select(0, va_idx_t),
                    Z_net_mask_t.index_select(0, va_idx_t),
                    U_seq_mask_t,
                    U_net_mask_t,
                ).detach().cpu().numpy().astype(np.float32)

            # tune alpha on FULL only
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                a = float(a)
                pred_s = apply_shrink(va_pred, va_base, a)
                sc = score_delta(va_true, pred_s)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = a

            # selection score
            va_s = apply_shrink(va_pred, va_base, a_best)
            va_score = score_delta(va_true, va_s)["score"]

            if va_score > best_score:
                best_score = float(va_score)
                best_alpha = float(a_best)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_raw = va_pred
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_raw

kf2 = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_raw = np.zeros_like(Y, dtype=np.float32)
oof_hit = np.zeros((N,), dtype=np.int32)

fold_scores = []
fold_alphas = []
fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(kf2.split(np.arange(N)), 1):
    best_score, best_alpha, best_epoch, best_state, best_va_raw = train_one_fold_string(tr_idx, va_idx, seed=SEED)

    fold_scores.append(float(best_score))
    fold_alphas.append(float(best_alpha))
    fold_epochs.append(int(best_epoch))

    oof_raw[va_idx] = best_va_raw
    oof_hit[va_idx] += 1

    print(f"[STRING] fold {fold}: best_score={best_score:.6f} alpha={best_alpha:.3f} epoch={best_epoch}")

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("[STRING] cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
EPOCHS_MED_S = int(np.median(fold_epochs))
print("[STRING] median best_epoch =", EPOCHS_MED_S)

# global alpha tuning (apply shrink ONCE to OOF)
best_global_alpha = 0.0
best_global_score = -1e18
for a in ALPHA_GRID:
    a = float(a)
    pred_a = apply_shrink(oof_raw, delta_baseline, a)
    sc = score_delta(Y, pred_a)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = a

oof_s = apply_shrink(oof_raw, delta_baseline, best_global_alpha)
oof_score = score_delta(Y, oof_s)["score"]

print("[STRING] OOF global alpha:", best_global_alpha, "OOF score:", float(oof_score))

ALPHA_SHRINK_STRING = float(best_global_alpha)

[STRING] fold 1: best_score=0.162130 alpha=0.860 epoch=35
[STRING] fold 2: best_score=0.124717 alpha=0.820 epoch=160
[STRING] fold 3: best_score=0.103056 alpha=0.700 epoch=70
[STRING] fold 4: best_score=0.176940 alpha=0.860 epoch=105
[STRING] fold 5: best_score=0.196951 alpha=0.860 epoch=125
[STRING] fold 6: best_score=0.190694 alpha=0.750 epoch=20
[STRING] fold 7: best_score=0.147440 alpha=0.700 epoch=15
[STRING] fold 8: best_score=0.108043 alpha=0.700 epoch=15
[STRING] cv mean: 0.151246437634452 std: 0.034151887723551634
[STRING] median best_epoch = 52
[STRING] OOF global alpha: 0.75 OOF score: 0.14813381830428263


[STRING] fold 1: best_score=0.162130 alpha=0.860 epoch=35
[STRING] fold 2: best_score=0.124717 alpha=0.820 epoch=160
[STRING] fold 3: best_score=0.103056 alpha=0.700 epoch=70
[STRING] fold 4: best_score=0.176940 alpha=0.860 epoch=105
[STRING] fold 5: best_score=0.196951 alpha=0.860 epoch=125
[STRING] fold 6: best_score=0.190694 alpha=0.750 epoch=20
[STRING] fold 7: best_score=0.147440 alpha=0.700 epoch=15
[STRING] fold 8: best_score=0.108043 alpha=0.700 epoch=15
[STRING] cv mean: 0.151246437634452 std: 0.034151887723551634
[STRING] median best_epoch = 52
[STRING] OOF global alpha: 0.75 OOF score: 0.14813381830428263

In [185]:
id_col = "pert_id" if "pert_id" in df_sub.columns else df_sub.columns[0]
sub_ids = df_sub[id_col].astype(str).tolist()

def pert_symbol_from_id(pid: str) -> str:
    return val_map.get(str(pid), str(pid))

sub_perts = [pert_symbol_from_id(pid) for pid in sub_ids]

def build_Z_base(perts):
    return np.vstack([emb_pert(p) for p in perts]).astype(np.float32)

def build_Z_string_and_masks(perts):
    Zs = np.zeros((len(perts), seq_dim), dtype=np.float32)
    Zn = np.zeros((len(perts), net_dim), dtype=np.float32)
    Ms = np.zeros((len(perts), 1), dtype=np.float32)
    Mn = np.zeros((len(perts), 1), dtype=np.float32)
    for i, p in enumerate(perts):
        gU = str(p).upper()
        vs = mean_emb_for_gene(gU, gene2prot, seq_idx, seq_emb_all)
        vn = mean_emb_for_gene(gU, gene2prot, net_idx, net_emb_all)
        if vs is not None:
            Zs[i] = vs
            Ms[i, 0] = 1.0
        if vn is not None:
            Zn[i] = vn
            Mn[i, 0] = 1.0
    return Zs, Zn, Ms, Mn

Z_base_sub = build_Z_base(sub_perts)
Z_seq_sub, Z_net_sub, Z_seq_sub_mask, Z_net_sub_mask = build_Z_string_and_masks(sub_perts)

Z_base_sub_t = torch.tensor(Z_base_sub, device=device, dtype=torch.float32)
Z_seq_sub_t  = torch.tensor(Z_seq_sub,  device=device, dtype=torch.float32)
Z_net_sub_t  = torch.tensor(Z_net_sub,  device=device, dtype=torch.float32)
Z_seq_sub_mask_t = torch.tensor(Z_seq_sub_mask, device=device, dtype=torch.float32)
Z_net_sub_mask_t = torch.tensor(Z_net_sub_mask, device=device, dtype=torch.float32)

In [186]:
def fit_full_model_string(seed: int, epochs_fixed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = HybridStringBilinearDeltaModel(
        d_base_p=Zt.shape[1],
        d_base_o=Uo_t.shape[1],
        d_seq=U_seq_t.shape[1],
        d_net=U_net_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
    ).to(device)
    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    idx = np.arange(N)
    for epoch in range(1, int(epochs_fixed) + 1):
        model.train()
        np.random.shuffle(idx)

        for start in range(0, N, BATCH_GENES):
            b = idx[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(
                Zt.index_select(0, b_t),
                Z_seq_t.index_select(0, b_t),
                Z_net_t.index_select(0, b_t),
                Uo_t,
                U_seq_t,
                U_net_t,
                Z_seq_mask_t.index_select(0, b_t),
                Z_net_mask_t.index_select(0, b_t),
                U_seq_mask_t,
                U_net_mask_t,
            )

            dt_b = Yt.index_select(0, b_t)
            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 10 == 0 or epoch == epochs_fixed:
            model.eval()
            with torch.no_grad():
                tr_pred = model(
                    Zt, Z_seq_t, Z_net_t,
                    Uo_t, U_seq_t, U_net_t,
                    Z_seq_mask_t, Z_net_mask_t,
                    U_seq_mask_t, U_net_mask_t
                ).detach().cpu().numpy().astype(np.float32)
                tr_pred_s = apply_shrink(tr_pred, delta_baseline, ALPHA_SHRINK_STRING)
                tr_score = score_delta(Y, tr_pred_s)["score"]
            print(f"[REFIT][TRAIN] seed={seed} epoch={epoch:4d} train_score={tr_score:.6f} loss={float(loss):.6f}")

    return model

epochs_fixed = int(EPOCHS_MED_S)
alpha_use = float(ALPHA_SHRINK_STRING)
print("[SUBMIT] epochs_fixed=", epochs_fixed, "alpha=", alpha_use, "seeds=", MODEL_SEEDS)

pred_list = []
for s in MODEL_SEEDS:
    m = fit_full_model_string(seed=int(s), epochs_fixed=epochs_fixed)
    m.eval()
    with torch.no_grad():
        p = m(
            Z_base_sub_t,
            Z_seq_sub_t,
            Z_net_sub_t,
            Uo_t,
            U_seq_t,
            U_net_t,
            Z_seq_sub_mask_t,
            Z_net_sub_mask_t,
            U_seq_mask_t,
            U_net_mask_t,
        ).detach().cpu().numpy().astype(np.float32)
    pred_list.append(p)

pred_raw = np.mean(np.stack(pred_list, axis=0), axis=0).astype(np.float32)  # (rows, G)

# IMPORTANT: baseline for submission must match submission rows.
# Use global mean-delta baseline broadcast, NOT per-pert D_train.
delta_baseline_vec = D_train.mean(axis=0).astype(np.float32)      # (G,)
sub_baseline = np.tile(delta_baseline_vec[None, :], (pred_raw.shape[0], 1)).astype(np.float32)  # (rows,G)

pred = apply_shrink(pred_raw, sub_baseline, alpha_use).astype(np.float32)

sub_out = df_sub.copy()
sub_out[gene_columns] = pred.astype(np.float64)

out_path = f"submission_hybrid_string_e{epochs_fixed}_a{alpha_use:.3f}.csv"
# sub_out.to_csv(out_path, index=False, float_format="%.20f")
print("[SUBMIT] wrote:", out_path, "shape:", sub_out.shape)

[SUBMIT] epochs_fixed= 52 alpha= 0.75 seeds= [6, 7, 8]
[REFIT][TRAIN] seed=6 epoch=  10 train_score=0.172801 loss=0.070335
[REFIT][TRAIN] seed=6 epoch=  20 train_score=0.231525 loss=0.069752
[REFIT][TRAIN] seed=6 epoch=  30 train_score=0.261455 loss=0.061344
[REFIT][TRAIN] seed=6 epoch=  40 train_score=0.283756 loss=0.062666
[REFIT][TRAIN] seed=6 epoch=  50 train_score=0.310746 loss=0.060297
[REFIT][TRAIN] seed=6 epoch=  52 train_score=0.317121 loss=0.053718
[REFIT][TRAIN] seed=7 epoch=  10 train_score=0.171356 loss=0.066581
[REFIT][TRAIN] seed=7 epoch=  20 train_score=0.233747 loss=0.069663
[REFIT][TRAIN] seed=7 epoch=  30 train_score=0.263195 loss=0.064849
[REFIT][TRAIN] seed=7 epoch=  40 train_score=0.288155 loss=0.065291
[REFIT][TRAIN] seed=7 epoch=  50 train_score=0.313980 loss=0.062484
[REFIT][TRAIN] seed=7 epoch=  52 train_score=0.320174 loss=0.060516
[REFIT][TRAIN] seed=8 epoch=  10 train_score=0.170447 loss=0.062575
[REFIT][TRAIN] seed=8 epoch=  20 train_score=0.226531 loss=0.